# Pipeline backend — Finance PWA

Analyse du pipeline réel de `backend/server.js` + `services/yahooEtfService.js`.  
Modèle HF : **Qwen/Qwen2.5-72B-Instruct** via `router.huggingface.co` (OpenAI-compatible).

Flux principal :
```
Yahoo Finance (yfinance) → prix clôture ×60j
    → 15 derniers prix  →  HF Router  →  Qwen2.5-72B  →  15 prix prédits
                          (si échec)  →  régression linéaire  →  30 points
```


In [ ]:
# Dépendances : pip install yfinance requests numpy
import os, json, re, math, sqlite3
from datetime import datetime, timedelta
from pathlib import Path

import requests
import numpy as np
import yfinance as yf

# Token HF — lit .env à la racine du projet
env_file = Path(__file__).parent.parent / ".env" if "__file__" in dir() else Path("../..") / ".env"
for line in env_file.read_text().splitlines():
    if line.startswith("HF_API_TOKEN="):
        os.environ.setdefault("HF_API_TOKEN", line.split("=", 1)[1].strip())

HF_TOKEN = os.environ.get("HF_API_TOKEN", "")
HF_URL   = "https://router.huggingface.co/v1/chat/completions"
MODEL_ID = "Qwen/Qwen2.5-72B-Instruct"

print("Token présent :", bool(HF_TOKEN))
print("Modèle        :", MODEL_ID)


## 1. Données Yahoo Finance

Réplique de `getEtfDetails(ticker)` dans `yahooEtfService.js` :
- `yahoo.chart(ticker, { period1, period2, interval: '1d' })` → `yf.Ticker.history()`
- Filtre les jours fériés (`close != null`)


In [ ]:
def get_historical(ticker: str, days: int = 365) -> list[dict]:
    """
    Équivalent JS :
        yahoo.chart(ticker, { period1, period2, interval: '1d' })
        .quotes.filter(q => q.close != null)
        .map(q => ({ date, open, high, low, close, volume, adjClose }))
    """
    end   = datetime.today()
    start = end - timedelta(days=days)

    hist = yf.Ticker(ticker).history(
        start=start.strftime("%Y-%m-%d"),
        end=end.strftime("%Y-%m-%d"),
        interval="1d",
        auto_adjust=True,
    )
    hist = hist.dropna(subset=["Close"])

    return [
        {
            "date"     : str(row.Index.date()),
            "open"     : round(float(row.Open),  2),
            "high"     : round(float(row.High),  2),
            "low"      : round(float(row.Low),   2),
            "close"    : round(float(row.Close), 2),
            "volume"   : int(row.Volume),
            "adjClose" : round(float(row.Close), 2),
        }
        for row in hist.itertuples()
    ]

# Test
SYMBOL = "CW8.PA"
historical = get_historical(SYMBOL, days=90)
print(f"{SYMBOL} — {len(historical)} jours récupérés")
print("Dernier :", historical[-1])


## 2. Pipeline prédiction — `/api/predict/stock`

Logique exacte du backend :

```js
// server.js l.332-373
const lastPrices = prices.slice(-15).map(p => Math.round(p)).join(', ')
const prompt = `The stock prices for the last 15 days are: ${lastPrices}.
  Predict the next 15 prices as a simple comma-separated list of numbers only.`

fetch('https://router.huggingface.co/v1/chat/completions', {
  model: 'Qwen/Qwen2.5-72B-Instruct',
  messages: [system, user],
  max_tokens: 100
})
```

Le backend extrait les nombres avec `/[\s,]+/` puis prend les 15 premiers.


In [ ]:
def hf_predict(prices: list[float], n_predict: int = 15) -> list[float] | None:
    """
    Appel HF Router — même payload que server.js l.339-353.
    Retourne None si pas de token ou erreur réseau.
    """
    if not HF_TOKEN:
        return None

    last_prices = ", ".join(str(round(p)) for p in prices[-15:])
    prompt = (
        f"The stock prices for the last 15 days are: {last_prices}. "
        f"Predict the next {n_predict} prices as a simple comma-separated "
        f"list of numbers only (e.g. 170, 172, 175, 178, 180...)."
    )

    payload = {
        "model"   : MODEL_ID,
        "messages": [
            {
                "role"   : "system",
                "content": "You are a financial forecasting assistant. "
                           "Always respond only with a list of 15 comma-separated numbers.",
            },
            {"role": "user", "content": prompt},
        ],
        "max_tokens": 100,
    }

    resp = requests.post(
        HF_URL,
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {HF_TOKEN}"},
        json=payload,
        timeout=30,
    )
    resp.raise_for_status()

    generated = resp.json()["choices"][0]["message"]["content"]
    print("[HF] réponse brute :", generated[:120])

    # Parsing identique à JS : split sur whitespace/virgules, parse float
    numbers = [float(x) for x in re.split(r"[\s,]+", generated) if x.replace(".", "").isdigit()]
    return numbers[:n_predict] if numbers else None


## 3. Fallback — régression linéaire

Utilisé quand `HF_API_TOKEN` absent **ou** erreur HF.  
Réplique exacte de server.js l.377-402 :

```js
// n points, calcul slope/intercept OLS
// stdDev des résidus → bruit gaussien × 1.5
// 30 points projetés
```


In [ ]:
def linear_regression_forecast(prices: list[float], n_forecast: int = 30) -> list[float]:
    """
    Régression OLS + bruit gaussien sur résidus.
    Identique au fallback JS (server.js l.377-402).
    """
    n = len(prices)
    x = np.arange(n, dtype=float)
    y = np.array(prices, dtype=float)

    # Formules OLS analytiques (même code que le JS)
    sum_x  = x.sum()
    sum_y  = y.sum()
    sum_xy = (x * y).sum()
    sum_xx = (x * x).sum()
    denom  = n * sum_xx - sum_x ** 2

    slope     = (n * sum_xy - sum_x * sum_y) / denom
    intercept = (sum_y - slope * sum_x) / n

    residuals = y - (slope * x + intercept)
    std_dev   = float(np.sqrt((residuals ** 2).mean()))

    rng = np.random.default_rng(seed=42)
    forecast = []
    for i in range(n_forecast):
        xi    = n + i
        trend = slope * xi + intercept
        noise = (rng.random() - 0.5) * std_dev * 1.5
        forecast.append(round(float(trend + noise), 2))

    return forecast


In [ ]:
def predict_stock(symbol: str) -> dict:
    """
    Réplique complète de GET /api/predict/stock (server.js l.307-448).
    """
    hist = get_historical(symbol, days=90)
    if len(hist) < 30:
        raise ValueError(f"Pas assez de données pour {symbol}")

    prices      = [h["close"] for h in hist[-60:]]   # 60 derniers jours
    current     = prices[-1]
    model_used  = "linear-regression"
    forecast    = []

    # Tentative HF
    hf_result = hf_predict(prices, n_predict=15)
    if hf_result:
        forecast   = hf_result
        model_used = MODEL_ID
    else:
        forecast   = linear_regression_forecast(prices, n_forecast=30)
        print("[predict] Fallback régression linéaire")

    return {
        "symbol"    : symbol,
        "prediction": {
            "current_price"  : current,
            "predicted_price": forecast[-1],
            "forecast"       : forecast,
            "history"        : prices,
            "confidence"     : 0.6 if "huggingface" in model_used else 0.75,
            "model"          : model_used,
        },
    }

result = predict_stock(SYMBOL)
pred   = result["prediction"]
print(f"Symbole       : {result['symbol']}")
print(f"Prix actuel   : {pred['current_price']}")
print(f"Prix prédit   : {pred['predicted_price']}")
print(f"Modèle        : {pred['model']}")
print(f"Confiance     : {pred['confidence']}")
print(f"Prévisions    : {pred['forecast'][:5]} ...")


## 4. Quiz — `/api/quiz/generate`

Même modèle, même prompt, même parsing JSON (server.js l.702-754).


In [ ]:
QUIZ_FALLBACK = [
    {"q": "Que signifie TER ?",
     "options": ["Total Expense Ratio", "Taux d'épargne réel", "Titre émis"],
     "correct": 0},
    {"q": "Qu'est-ce que le DCA ?",
     "options": ["Un ETF", "Investir régulièrement", "Un indice"],
     "correct": 1},
]

def generate_quiz() -> list[dict]:
    """
    Réplique de GET /api/quiz/generate (server.js l.702-754).
    Utilise Qwen2.5-72B — fallback statique si erreur.
    """
    if not HF_TOKEN:
        print("[quiz] Pas de token → fallback statique")
        return QUIZ_FALLBACK

    prompt = (
        "Génère 5 questions de quiz sur la finance "
        "(investissement, bourse, crypto, fiscalité française). "
        "Pour chaque question, donne 3 options et l'index de la bonne réponse (0, 1 ou 2). "
        "Réponds uniquement au format JSON strict, sans texte avant ou après, comme ceci : "
        '[{"q": "Ma question ?", "options": ["Rép A", "Rép B", "Rép C"], "correct": 0}, ...]'
    )

    payload = {
        "model"      : MODEL_ID,
        "messages"   : [
            {"role": "system",
             "content": "Tu es un expert en éducation financière. "
                        "Tu réponds uniquement en JSON valide."},
            {"role": "user", "content": prompt},
        ],
        "max_tokens" : 1000,
        "temperature": 0.7,
    }

    try:
        resp = requests.post(
            HF_URL,
            headers={"Content-Type": "application/json",
                     "Authorization": f"Bearer {HF_TOKEN}"},
            json=payload,
            timeout=30,
        )
        resp.raise_for_status()
        content = resp.json()["choices"][0]["message"]["content"]

        # Nettoyage balises ```json (même logique que server.js l.742)
        json_str   = re.sub(r"```json|```", "", content).strip()
        questions  = json.loads(json_str)
        return questions
    except Exception as e:
        print(f"[quiz] Erreur HF ({e}) → fallback statique")
        return QUIZ_FALLBACK

questions = generate_quiz()
for i, q in enumerate(questions, 1):
    print(f"{i}. {q['q']}")
    for j, opt in enumerate(q.get('options', [])):
        mark = "✓" if j == q.get('correct') else " "
        print(f"   [{mark}] {opt}")
    print()


## 5. Scoring ETF — `calcMatch` + volatilité

Fonctions purement algorithmiques de `yahooEtfService.js`.


In [ ]:
def calc_match(ter: float, esg: str) -> int:
    """
    Score 0-100 : 40 % TER + 60 % ESG.
    Copie exacte de calcMatch() — yahooEtfService.js l.318-323.
    """
    ter_score = max(0, 100 - ter * 50)
    esg_order = ["B", "A", "AA", "AAA"]
    esg_idx   = esg_order.index(esg) if esg in esg_order else -1
    esg_score = 60 + esg_idx * 13 if esg_idx >= 0 else 70
    return round(ter_score * 0.4 + esg_score * 0.6)


def calculate_volatility(closes: list[float]) -> float:
    """
    Volatilité annualisée (%).
    Copie exacte de calculateVolatility() — yahooEtfService.js l.327-339.
    """
    if len(closes) < 2:
        return 0.0
    returns  = [(closes[i] - closes[i-1]) / closes[i-1] for i in range(1, len(closes))]
    mean     = sum(returns) / len(returns)
    variance = sum((r - mean) ** 2 for r in returns) / len(returns)
    return round(math.sqrt(variance * 252) * 100, 2)


# Exemple avec les données déjà récupérées
closes = [h["close"] for h in historical]
vol    = calculate_volatility(closes)

print(f"Volatilité annualisée {SYMBOL} : {vol} %")
print()
print("Scores ETF (ter, esg) → match :")
for ter, esg in [(0.2, "AAA"), (0.5, "AA"), (1.0, "A"), (2.0, "B")]:
    print(f"  TER={ter}  ESG={esg}  →  {calc_match(ter, esg)}/100")


## 6. Base SQLite — `backend/data/finance.db`

La même base que le backend utilise en dev (better-sqlite3 côté JS → sqlite3 côté Python).


In [ ]:
DB_PATH = Path("../../backend/data/finance.db")

if DB_PATH.exists():
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()

    # Tables
    tables = cur.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()
    print("Tables :", [t[0] for t in tables])

    # Nb lignes par table
    for (tbl,) in tables:
        count = cur.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
        print(f"  {tbl:<20} {count} lignes")

    con.close()
else:
    print(f"DB non trouvée : {DB_PATH.resolve()}")
    print("Lance `npm run dev:all` une fois pour créer la base.")


## 7. Pipeline complet — récap

```
GET /api/predict/stock?symbol=CW8.PA
          │
          ▼
    get_historical()          ← yfinance / yahoo.chart()
    60 derniers jours closes
          │
          ▼
    hf_predict()              ← POST router.huggingface.co/v1/chat/completions
    model: Qwen/Qwen2.5-72B   ← prompt: "last 15 prices → predict 15"
    parse CSV numbers
          │ (échec / pas de token)
          ▼
    linear_regression_forecast()
    OLS + bruit gaussien × 1.5
          │
          ▼
    { current_price, predicted_price, forecast[15|30], model, confidence }
```
